In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
DATA_PATH = kagglehub.competition_download('playground-series-s6e3')

print('Data source import complete.')

In [ ]:
ORG_DATA_PATH = kagglehub.dataset_download("cdeotte/s6e3-original-dataset")

print("Path to dataset files:", ORG_DATA_PATH)

## FEAT. ENG.

In [ ]:
!pip install catboost -qq

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib
import warnings

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', 2000)
pd.set_option('display.max_rows', 2000)
pd.set_option('future.no_silent_downcasting', True)
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv(DATA_PATH + '/train.csv')
test = pd.read_csv(DATA_PATH + '/test.csv')
sub = pd.read_csv(DATA_PATH + '/sample_submission.csv')

org_train = pd.read_csv(ORG_DATA_PATH + '/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
len(train), len(org_train)

In [ ]:
train.head()

In [ ]:
org_train.head()

In [ ]:
train.info()

In [ ]:
org_train.info()

In [ ]:
org_train["TotalCharges"] = pd.to_numeric(org_train["TotalCharges"], errors='coerce')
org_train.rename(columns={"customerID": "id"}, inplace=True)
org_train.info()

In [ ]:
cat_cols = [col for col in train.columns if train[col].dtype=="object" and col not in ["id", "Churn"]]
num_cols = [col for col in train.columns if col not in cat_cols and col not in ["id", "Churn"]]

for col in cat_cols:
  train[col] = train[col].astype("category")
  test[col] = test[col].astype("category")
  org_train[col] = org_train[col].astype("category")

In [ ]:
TARGET = "Churn"
FEATURES = [col for col in train.columns if col not in ["id", "Churn"]]

le = LabelEncoder()
le.fit(pd.concat([train[TARGET], org_train[TARGET]], axis=0))
org_train[TARGET] = le.transform(org_train[TARGET])
train[TARGET] = le.transform(train[TARGET])

print(len(FEATURES))

In [ ]:

orig_mean = org_train[TARGET].mean()

orig_te_cols = []
for col in cat_cols:
    te = org_train.groupby(col)[TARGET].mean().reset_index()
    te.columns = [col, f"TE_{col}_orig"]
    train = train.merge(te, on=col, how="left")
    test  = test.merge(te, on=col, how="left")
    train[f"TE_{col}_orig"].fillna(orig_mean, inplace=True)
    test[f"TE_{col}_orig"].fillna(orig_mean, inplace=True)
    orig_te_cols.append(f"TE_{col}_orig")

FEATURES += orig_te_cols
print(len(FEATURES))

In [ ]:
# def digit_features(df):

#     DIGIT = []
#     target_cols_for_digits = ['tenure', 'MonthlyCharges', 'TotalCharges']
#     for col in target_cols_for_digits:
#         max_val = df[col].abs().max()

#         if max_val == 0:
#             max_int_digits = 1
#         else:
#             max_int_digits = int(np.log10(max_val)) + 1

#         for i in range(max_int_digits):
#             new_col = f"{col}_digit_pos_{i+1}"

#             df[new_col] = (df[col] // (10**i)) % 10
#             df[new_col] = df[new_col].astype(int)

#             DIGIT.append(new_col)

#         if df[col].dtype == 'float64':
#             max_dec_digits = 2

#             if (df[col] % 1 != 0).any():
#                 for i in range(1, max_dec_digits + 1):
#                     new_col = f"{col}_digit_dec_{i}"

#                     df[new_col] = (df[col] * (10**i)).round().astype(int) % 10

#                     DIGIT.append(new_col)

#     print(f"{len(DIGIT)} Digit Features Created!!")
#     print(f"Sample features: {DIGIT[:5]} ...")

#     return df, DIGIT

# train, DIGIT = digit_features(train)
# test, _ = digit_features(test)
# FEATURES += DIGIT
# print(len(FEATURES))

In [ ]:
train.head()

In [ ]:

train[TARGET].value_counts()

In [ ]:
for df in [train, test, org_train]:
    df['charges_deviation']      = (df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    df['avg_monthly_charges']    = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')

FEATURES += ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges']

SERVICE_COLS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for df in [train, test, org_train]:
    df['service_count'] = (df[SERVICE_COLS] == 'Yes').sum(axis=1).astype('float32')

FEATURES += ['service_count']
print(len(FEATURES))

In [ ]:
train.head()

In [ ]:
DIGIT_FEATURES = [
    # Tenure
    'tenure_first_digit','tenure_last_digit','tenure_second_digit',
    'tenure_mod10','tenure_mod12','tenure_num_digits',
    'tenure_is_multiple_10','tenure_rounded_10','tenure_dev_from_round10',

    # MonthlyCharges
    'mc_first_digit','mc_last_digit','mc_second_digit',
    'mc_mod10','mc_mod100','mc_num_digits',
    'mc_is_multiple_10','mc_is_multiple_50',
    'mc_rounded_10','mc_fractional','mc_dev_from_round10',

    # TotalCharges
    'tc_first_digit','tc_last_digit','tc_second_digit',
    'tc_mod10','tc_mod100','tc_num_digits',
    'tc_is_multiple_10','tc_is_multiple_100',
    'tc_rounded_100','tc_fractional','tc_dev_from_round100',

    # Derived
    'tenure_years','tenure_months_in_year',
    'mc_per_digit','tc_per_digit'
]

for df in [train, test]:

    # Tenure digits
    t_str = df['tenure'].astype(str)
    df['tenure_first_digit'] = t_str.str[0].astype(int)
    df['tenure_last_digit'] = t_str.str[-1].astype(int)
    df['tenure_second_digit'] = t_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)

    df['tenure_mod10'] = df['tenure'] % 10
    df['tenure_mod12'] = df['tenure'] % 12
    df['tenure_num_digits'] = t_str.str.len()

    df['tenure_is_multiple_10'] = (df['tenure'] % 10 == 0).astype('float32')

    df['tenure_rounded_10'] = np.round(df['tenure']/10)*10
    df['tenure_dev_from_round10'] = abs(df['tenure'] - df['tenure_rounded_10'])

    # MonthlyCharges
    mc_str = df['MonthlyCharges'].astype(str).str.replace('.', '')

    df['mc_first_digit'] = mc_str.str[0].astype(int)
    df['mc_last_digit'] = mc_str.str[-1].astype(int)
    df['mc_second_digit'] = mc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)

    df['mc_mod10'] = np.floor(df['MonthlyCharges']) % 10
    df['mc_mod100'] = np.floor(df['MonthlyCharges']) % 100

    df['mc_num_digits'] = np.floor(df['MonthlyCharges']).astype(int).astype(str).str.len()

    df['mc_is_multiple_10'] = (np.floor(df['MonthlyCharges']) % 10 == 0).astype('float32')
    df['mc_is_multiple_50'] = (np.floor(df['MonthlyCharges']) % 50 == 0).astype('float32')

    df['mc_rounded_10'] = np.round(df['MonthlyCharges']/10)*10
    df['mc_fractional'] = df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])
    df['mc_dev_from_round10'] = abs(df['MonthlyCharges'] - df['mc_rounded_10'])

    # TotalCharges
    tc_str = df['TotalCharges'].astype(str).str.replace('.', '')

    df['tc_first_digit'] = tc_str.str[0].astype(int)
    df['tc_last_digit'] = tc_str.str[-1].astype(int)
    df['tc_second_digit'] = tc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)

    df['tc_mod10'] = np.floor(df['TotalCharges']) % 10
    df['tc_mod100'] = np.floor(df['TotalCharges']) % 100

    df['tc_num_digits'] = np.floor(df['TotalCharges']).astype(int).astype(str).str.len()

    df['tc_is_multiple_10'] = (np.floor(df['TotalCharges']) % 10 == 0).astype('float32')
    df['tc_is_multiple_100'] = (np.floor(df['TotalCharges']) % 100 == 0).astype('float32')

    df['tc_rounded_100'] = np.round(df['TotalCharges']/100)*100
    df['tc_fractional'] = df['TotalCharges'] - np.floor(df['TotalCharges'])

    df['tc_dev_from_round100'] = abs(df['TotalCharges'] - df['tc_rounded_100'])

    # Derived
    df['tenure_years'] = df['tenure'] // 12
    df['tenure_months_in_year'] = df['tenure'] % 12

    df['mc_per_digit'] = df['MonthlyCharges']/(df['mc_num_digits']+0.001)
    df['tc_per_digit'] = df['TotalCharges']/(df['tc_num_digits']+0.001)

FEATURES += DIGIT_FEATURES
print(f"    Created {len(DIGIT_FEATURES)} digital Numerical features")
print(len(FEATURES))

In [ ]:
# 5. Distribution Features
print("[5/7] Creating Distribution Features (percentile ranks, z-scores)...")

def pctrank_against(values, reference):
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    mu, sigma = np.mean(reference), np.std(reference)
    return (np.zeros(len(values), dtype='float32') if sigma == 0
            else ((values - mu) / sigma).astype('float32'))

orig_churner_tc    = org_train.loc[org_train[TARGET] == 1, 'TotalCharges'].values
orig_nonchurner_tc = org_train.loc[org_train[TARGET] == 0, 'TotalCharges'].values
orig_tc            = org_train['TotalCharges'].values
orig_is_mc_mean    = org_train.groupby('InternetService')['MonthlyCharges'].mean()

for df in [train, test]:
    tc = df['TotalCharges'].values
    df['pctrank_nonchurner_TC']  = pctrank_against(tc, orig_nonchurner_tc)
    df['pctrank_churner_TC']     = pctrank_against(tc, orig_churner_tc)
    df['pctrank_orig_TC']        = pctrank_against(tc, orig_tc)
    df['zscore_churn_gap_TC'] = (np.abs(zscore_against(tc, orig_churner_tc)) -
                                 np.abs(zscore_against(tc, orig_nonchurner_tc))).astype('float32')
    df['zscore_nonchurner_TC'] = zscore_against(tc, orig_nonchurner_tc)
    df['pctrank_churn_gap_TC'] = (pctrank_against(tc, orig_churner_tc) -
                                  pctrank_against(tc, orig_nonchurner_tc)).astype('float32')
    # Convert to float series explicitly to avoid categorical constraints
    is_means = df['InternetService'].map(orig_is_mc_mean).astype('float32')
    df['resid_IS_MC'] = (df['MonthlyCharges'] - is_means.fillna(0)).astype('float32')
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in org_train['InternetService'].unique():
        mask = df['InternetService'] == cat_val
        ref = org_train.loc[org_train['InternetService'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['cond_pctrank_IS_TC'] = vals

    vals = np.zeros(len(df), dtype='float32')
    # Corrected loop logic
    for cat_val in org_train['Contract'].unique():
        mask = df['Contract'] == cat_val
        ref = org_train.loc[org_train['Contract'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.any(): # Use .any() on the mask
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['cond_pctrank_C_TC'] = vals

DIST_FEATURES = [
    'pctrank_nonchurner_TC', 'zscore_churn_gap_TC', 'pctrank_churn_gap_TC',
    'resid_IS_MC', 'cond_pctrank_IS_TC', 'zscore_nonchurner_TC',
    'pctrank_orig_TC', 'pctrank_churner_TC', 'cond_pctrank_C_TC'
]

FEATURES += DIST_FEATURES
print(len(FEATURES))

In [ ]:
# 6. Quantile Distance Features
print("[6/7] Creating Quantile Distance Features...")
for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
    ch_q = np.quantile(orig_churner_tc, q_val)
    nc_q = np.quantile(orig_nonchurner_tc, q_val)
    for df in [train, test]:
        df[f'dist_To_ch_{q_label}'] = np.abs(df['TotalCharges'] - ch_q).astype('float32')
        df[f'dist_To_nc_{q_label}'] = np.abs(df['TotalCharges'] - nc_q).astype('float32')
        df[f'qdist_gap_To_{q_label}'] = (df[f'dist_To_nc_{q_label}'] - df[f'dist_To_ch_{q_label}']).astype('float32')

QDIST_FEATURES = [
    'qdist_gap_To_q50', 'dist_To_ch_q50', 'dist_To_nc_q50',
    'dist_To_nc_q25', 'qdist_gap_To_q25',
    'dist_To_nc_q75', 'dist_To_ch_q75', 'qdist_gap_To_q75'
]
FEATURES += QDIST_FEATURES
print(f"    Created {len(QDIST_FEATURES)} quantile distance features")
print(len(FEATURES))

In [ ]:
FOLDS = 5
# SEED_LIST = [12, 22, 32, 42, 52]
SEED_LIST = [42]

## TABICL

In [ ]:
%%time
!pip install -q tabicl

In [ ]:
%%time

import os
import random
import warnings
import numpy as np, pandas as pd
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch

from tabicl import TabICLClassifier

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)


train = pd.read_csv(DATA_PATH + '/train.csv')
test = pd.read_csv(DATA_PATH + '/test.csv')
sub = pd.read_csv(DATA_PATH + '/sample_submission.csv')

org_train = pd.read_csv(ORG_DATA_PATH + '/WA_Fn-UseC_-Telco-Customer-Churn.csv')

ID = 'id'
TARGET = 'Churn'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Contract', 'InternetService', 'PaymentMethod'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_MonthlyCharges_/_TotalCharges'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)).astype('float32')
    df['_TotalCharges_/_tenure']  = (df['TotalCharges'] / (df['tenure'] + 1e-6)).astype('float32')
    df['_Monthly_to_avg_ratio'] = (df['MonthlyCharges'] / (df['_TotalCharges_/_tenure'] + 1e-6)).astype('float32')
    df['_TotalCharges_/_MonthlyCharges']  = (df['TotalCharges'] / (df['MonthlyCharges'] + 1e-6)).astype('float32')
    df['_tenure_sq'] = (df['tenure'] ** 2).astype("float32")

    # Digit extraction
    for col in ['TotalCharges']:
        k = -3
        digit_name = f"{col}_d{k}_"
        df[digit_name] = ((df[col] * 10**k) % 10).astype('int8')
        df[digit_name] = df[digit_name].astype('category')

    # Discretize numericals
    bin_config = {'TotalCharges': [4000, 500], 'MonthlyCharges': [200, 100]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_bin_"
            if fit:
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode='ordinal',
                    strategy='quantile',
                    subsample=None
                )
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype('int32')
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        round_level = 0
        if fit:
            round_flag = col == 'TotalCharges'
            series = df[col].round(round_level) if round_flag else df[col]
            codes, uniques = series.factorize()
            category_map[col] = {'uniques': uniques, 'round_flag': round_flag}
        else:
            round_flag = category_map[col]['round_flag']
            uniques = category_map[col]['uniques']
            series = df[col].round(round_level) if round_flag else df[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = series.map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Create interaction categories
    combo_names = []
    for col1, col2, col3 in important_combos:
        combo_name = f"{col1}_{col2}_{col3}_"
        combo_names.append(combo_name)
        combo_series = df[col1].astype(str) + '_' + df[col2].astype(str) + '_' + df[col3].astype(str)
        if fit:
            codes, uniques = combo_series.factorize()
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

class CFG:
    FOLDS = 5
    SEED = 42

skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

from sklearn.base import BaseEstimator, ClassifierMixin, clone
import gc

class VotingClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, params):
        self.params =  params
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.X_ = X.copy()
        self.y_ = y.copy()
        return self
    def predict_proba(self, X):
        probs = []
        for params in self.params:
            model = TabICLClassifier(**params).fit(self.X_, self.y_)
            probs.append(model.predict_proba(X))
            del model
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        return np.mean(probs, axis=0)

n_models = 4
params = [
    {
        'device': 'cuda',
        'n_estimators': 2,
        'kv_cache': True,
        'random_state': i
    } for i in range(n_models)
]


for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print("#"*16)
    print(f"### Fold {fold}/{CFG.FOLDS} ...")
    print("#"*16)

    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Target encoding
    te_cols = combo_names
    TE = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
    tr_enc = TE.fit_transform(X_tr[te_cols], y.iloc[tr_idx])
    val_enc = TE.transform(X_val[te_cols])
    tst_enc = TE.transform(X_tst[te_cols])

    te_names = [f"_{col}TE" for col in te_cols]
    X_tr[te_names] = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    model = VotingClassifier(params)
    model.fit(
            X_tr, y.iloc[tr_idx],
        )

    val_preds = model.predict_proba(X_val)[:, 1]
    fold_test_preds = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / CFG.FOLDS

    fold_score = roc_auc_score(y.iloc[val_idx], val_preds)
    print(f"Fold {fold} AUC: {fold_score:.6f}")
    torch.cuda.empty_cache()


oof_score = roc_auc_score(y, oof_preds)
print("\n" + "="*24)
print(f"OOF AUC: {oof_score:.6f}")
print("="*24)

oof_df = pd.DataFrame({ID: train_id, TARGET: oof_preds})
oof_df.to_csv('oof_preds.csv', index=False)

sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
from google.colab import files

files.download('oof_preds.csv')
files.download('submission.csv')

## XGBOOST RESIDUALS

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --- 1. Helper Functions ---
def prob_to_logit(p, eps=1e-6):
    """Converts probabilities to log-odds (logits) for XGBoost base_margin."""
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def clean_for_lr(df):
    """Cleans data for Logistic Regression: handles Inf and NaN."""
    return df.replace([np.inf, -np.inf], np.nan).fillna(0)

# --- 2. Configuration & Features ---
SEED_LIST = [42]
FOLDS = 5
TARGET = "Churn"

# Identify column types
cat_cols = [col for col in train.columns if train[col].dtype.name == 'category' or train[col].dtype == 'object']
# Make sure we don't include ID or Target in features
FEATURES = [col for col in train.columns if col not in ["id", TARGET]]

# --- 3. Prep Data for Logistic Regression (Numeric Only) ---
# LR needs One-Hot Encoding and no NaNs
X_lr_train = pd.get_dummies(train[FEATURES], columns=[c for c in cat_cols if c in FEATURES], drop_first=True)
X_lr_test = pd.get_dummies(test[FEATURES], columns=[c for c in cat_cols if c in FEATURES], drop_first=True)

# Align columns between train and test
X_lr_train, X_lr_test = X_lr_train.align(X_lr_test, join='left', axis=1, fill_value=0)

# Clean for LR
X_lr_train = clean_for_lr(X_lr_train)
X_lr_test = clean_for_lr(X_lr_test)

# --- 4. Model Parameters ---
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.01,
    "max_depth": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "seed": 42,
    "device": "cuda", # Change to "cpu" if no GPU available
    "enable_categorical": True,
}

# --- 5. Training Loop ---
y = train[TARGET].astype(int)
X_xgb_full = train[FEATURES]

oof = np.zeros(len(train), dtype=np.float32)
test_pred = np.zeros(len(test), dtype=np.float32)

for SEED in SEED_LIST:
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
    seed_oof = np.zeros(len(train), dtype=np.float32)
    seed_test = np.zeros(len(test), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_xgb_full, y), 1):
        # Split Data
        X_tr_lr, X_va_lr = X_lr_train.iloc[tr_idx], X_lr_train.iloc[va_idx]
        X_tr_xgb, X_va_xgb = X_xgb_full.iloc[tr_idx], X_xgb_full.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        # A. Logistic Regression Pipeline (Scaler is vital for LR convergence)
        lr_model = make_pipeline(StandardScaler(), LogisticRegression(        penalty="l2",
        C=0.5,
        max_iter=4000,
        tol=1e-4,
        fit_intercept=True,
        verbose=0,))
        lr_model.fit(X_tr_lr, y_tr)

        # B. Generate Base Margins (Logits)
        bm_tr = prob_to_logit(lr_model.predict_proba(X_tr_lr)[:, 1])
        bm_va = prob_to_logit(lr_model.predict_proba(X_va_lr)[:, 1])
        bm_te = prob_to_logit(lr_model.predict_proba(X_lr_test)[:, 1])

        # C. XGBoost Training
        dtr = xgb.DMatrix(X_tr_xgb, label=y_tr, enable_categorical=True)
        dva = xgb.DMatrix(X_va_xgb, label=y_va, enable_categorical=True)
        dte = xgb.DMatrix(test[FEATURES], enable_categorical=True)

        dtr.set_base_margin(bm_tr)
        dva.set_base_margin(bm_va)
        dte.set_base_margin(bm_te)

        booster = xgb.train(
            params=xgb_params,
            dtrain=dtr,
            num_boost_round=5000,
            evals=[(dva, "valid")],
            early_stopping_rounds=200,
            verbose_eval=200
        )

        # D. OOF and Test Predictions
        va_pred = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
        seed_oof[va_idx] = va_pred

        seed_test += booster.predict(dte, iteration_range=(0, booster.best_iteration + 1)) / FOLDS

        print(f"SEED {SEED} Fold {fold} AUC: {roc_auc_score(y_va, va_pred):.6f}")

    print(f"--- SEED {SEED} COMPLETE | OOF AUC: {roc_auc_score(y, seed_oof):.6f} ---")
    oof += seed_oof / len(SEED_LIST)
    test_pred += seed_test / len(SEED_LIST)

print(f"\nOVERALL OOF AUC: {roc_auc_score(y, oof):.6f}")


In [ ]:
np.save('xgb_res_oof_preds.npy', oof)


# Save Submission
sub[TARGET] = test_pred
sub.to_csv("xgb_res_submission.csv", index=False)

## HISTGBM

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

import cuml.accel
cuml.accel.install()


skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

hgb_oof = np.zeros(len(train), dtype=np.float32)
hgb_test = np.zeros(len(test), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train[TARGET]), 1):

    X_train = train.iloc[tr_idx][FEATURES].copy()
    y_train = train.iloc[tr_idx][TARGET]

    X_valid = train.iloc[va_idx][FEATURES].copy()
    y_valid = train.iloc[va_idx][TARGET]


    X_test = test[FEATURES].copy()

    hgb = HistGradientBoostingClassifier(max_iter=3000, max_depth=5, random_state=42)
    hgb.fit(X_train, y_train)

    hgb_oof[va_idx] = hgb.predict_proba(X_valid)[:, 1].astype(np.float32)
    hgb_test += hgb.predict_proba(X_test)[:, 1].astype(np.float32) / FOLDS

    fold_auc = roc_auc_score(y_valid, hgb_oof[va_idx])
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

print("hgb OOF AUC:", roc_auc_score(train[TARGET], hgb_oof))
np.save("hgb_oof_preds", hgb_oof)

sub = pd.read_csv(DATA_PATH + '/sample_submission.csv')

sub['Churn'] = hgb_test
sub.to_csv("hgb_submission.csv",index=False)
sub.head()

## 0.9173

## XGBOOST

In [ ]:
import xgboost as xgb

print(f"XGBoost version {xgb.__version__}")
from cuml.preprocessing import TargetEncoder


In [ ]:
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.01,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.6,
    "seed": 42,
    "device": "cuda",
    "enable_categorical": True,
}


In [ ]:
xgb_final_oof_preds = np.zeros(len(train))
xgb_final_test_preds = np.zeros(len(test))


for SEED in SEED_LIST:
    print(f"############ SEED {SEED} ############")
    xgb_oof_preds = np.zeros(len(train))
    xgb_test_preds = np.zeros(len(test))

    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
    for fold, (train_idx, val_idx) in enumerate(skf.split(train[FEATURES], train[TARGET])):
        print("#"*25)
        print(f"### Fold {fold+1} ###")
        print("#"*25)

        X_train = train.iloc[train_idx][FEATURES].copy()
        y_train = train.iloc[train_idx][TARGET]

        X_valid = train.iloc[val_idx][FEATURES].copy()
        y_valid = train.iloc[val_idx][TARGET]
        X_test = test[FEATURES].copy()

        cat_features = [c for c in X_train.columns if c in cat_cols]


        dtr = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dva = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
        dte = xgb.DMatrix(X_test, enable_categorical=True)

        print("Training an XGBoost Model...")
        model = xgb.train(
            params=xgb_params,
            dtrain=dtr,
            num_boost_round=5000,
            evals=[(dtr, "train"), (dva, "valid")],
            early_stopping_rounds=600,
            verbose_eval=200,

        )
        print("Training complete.")


        val_pred = model.predict(dva, iteration_range=(0, model.best_iteration + 1))
        xgb_oof_preds[val_idx] = val_pred

        xgb_test_preds += model.predict(dte, iteration_range=(0, model.best_iteration + 1)) / FOLDS

        fold_auc = roc_auc_score(y_valid, val_pred)
        print(f"Fold AUC: {fold_auc:.6f}")

    print(f"SEED {SEED} OOF AUC: {roc_auc_score(train[TARGET], xgb_oof_preds):.6f}")
    xgb_final_oof_preds += xgb_oof_preds / len(SEED_LIST)
    xgb_final_test_preds += xgb_test_preds / len(SEED_LIST)

print("Overall OOF AUC:", roc_auc_score(train[TARGET], xgb_final_oof_preds))

# 0.955678

## CATBOOST

In [ ]:
import catboost
from catboost import CatBoostClassifier

from cuml.preprocessing import TargetEncoder

print(catboost.__version__)

In [ ]:
catboost_params = {
    'iterations': 5000,
    'learning_rate': 0.02,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 200,
    # 'border_count': 254,
    'task_type':'GPU',
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'scale_pos_weight': 2.0,
}

In [ ]:
%%time

cat_final_oof_preds = np.zeros(len(train))
cat_final_test_preds = np.zeros(len(test))


for SEED in SEED_LIST:
    print(f"############ SEED {SEED} ############")
    cat_oof_preds = np.zeros(len(train))
    cat_test_preds = np.zeros(len(test))

    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
    for fold, (train_idx, val_idx) in enumerate(skf.split(train[FEATURES], train[TARGET])):
        print("#"*25)
        print(f"### Fold {fold+1} ###")
        print("#"*25)

        X_train = train.iloc[train_idx][FEATURES].copy()
        y_train = train.iloc[train_idx][TARGET]

        X_valid = train.iloc[val_idx][FEATURES].copy()
        y_valid = train.iloc[val_idx][TARGET]
        X_test = test[FEATURES].copy()

        cat_features = [c for c in X_train.columns if c in cat_cols]

        # CC = [c for c in FEATURES if (not c.startswith("TE_"))]
        # stats = ["mean"]
        # for stat in stats:
        #   print(f"Target encoding {len(CC)} features... ", end="")
        #   for i, c in enumerate(CC):
        #       if i % 5 == 0: print(f"{i}, ", end="")
        #       n = f"TE_{c}_{stat}"
        #       # TE0 = TargetEncoder(cols=[c], handle_missing='value', handle_unknown='value')
        #       TE0 = TargetEncoder(n_folds=10, smooth=4, split_method='random', stat=stat)
        #       X_train[n] = TE0.fit_transform(X_train[[c]], y_train).astype('float32')
        #       X_valid[n] = TE0.transform(X_valid[[c]]).astype('float32')
        #       X_test[n] = TE0.transform(X_test[[c]]).astype('float32')
        #   print()

        # if np.any(np.isnan(X_train)) or np.any(np.isinf(X_train)):
        #   print(f"WARNING: Found NaN/Inf values in input data!")
        #   X_train.fillna(0, inplace=True)
        # if np.any(np.isnan(X_valid)) or np.any(np.isinf(X_valid)):
        #   print(f"WARNING: Found NaN/Inf values in input data!")
        #   X_valid.fillna(0, inplace=True)
        # if np.any(np.isnan(X_test)) or np.any(np.isinf(X_test)):
        #   print(f"WARNING: Found NaN/Inf values in input data!")
        #   X_test.fillna(0, inplace=True)

        model = CatBoostClassifier(**catboost_params)
        print("Training a CatBoost Model...")
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            use_best_model=True,
            cat_features=cat_features,
        )
        print("Training complete.")

        cat_oof_preds[val_idx] = model.predict_proba(X_valid)[:, 1]
        cat_test_preds += model.predict_proba(X_test)[:, 1] / FOLDS

    cat_final_oof_preds += cat_oof_preds / len(SEED_LIST)
    cat_final_test_preds += cat_test_preds / len(SEED_LIST)


overall_oof = roc_auc_score(train[TARGET], cat_final_oof_preds)
print(f" CATBOOST Overall CV = {overall_oof}")
np.save('cat_oof_preds.npy', cat_final_oof_preds)

#0.91778

In [ ]:
import os

sub = pd.read_csv(DATA_PATH + '/sample_submission.csv')

sub['Churn'] = cat_final_test_preds
sub.to_csv("cat_submission.csv",index=False)
sub.head()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions submit -c playground-series-s6e3 -f cat_submission.csv  -m "catboost lossguide. cv: {overall_oof}"

## HILL CLIMBING

In [ ]:
import numpy as np
import cupy as cp
import gc
from sklearn.metrics import roc_auc_score

In [ ]:

# cat_te_depthwise_oof = pd.read_csv('/content/cat_te_depthwise_oof.csv')
# np.save('cat_te_depthwise_oof_preds.npy', cat_te_depthwise_oof["Churn"])


# xgb_deneme_oof = pd.read_csv('/content/oof_predictions.csv')
# np.save('/content/drive/MyDrive/Colab Notebooks/KAGGLE PS S6E3/OOF/xgb_deneme_oof_preds.npy', xgb_deneme_oof["pred_lgb"])

In [ ]:
files = []
x_train = []
PATH = "/content/drive/MyDrive/Colab Notebooks/KAGGLE PS S6E3/OOF"

print("Loading files...")
for c in ['cat', 'nn', 'xgb', 'lr', 'xgb_pseudo', 'xgb_te', 'xgb_v2', 'torch_frame', 'realmlp', 'tabm', 'cat_v2', 'resnet', 'lgbm_te', 'cat_te', 'tab_transformer',
          'lgbm_te_goss', 'gnn', 'xgb_res', 'hgb', 'ftt', 'bartz', 'tabm_pseudo', 'tabm_v2', 'cat_emb', 'hgb_te', 'cat_te_ordered', 'tabm_te', 'xlearn',
          'xgb_te_res', 'realmlp_32', 'realmlp_scratch', 'ftt_scratch', 'gatenet_scratch', 'danet_scratch', 'dcnv2_scratch', 'tabm_scratch']:
    print(f"=> {c} ",end="")
    oof = np.load(f"{PATH}/{c}_oof_preds.npy")
    x_train.append(oof)
    print(oof.shape)
    files.append(f"{c}_oof")
    print()


In [ ]:
x_train = np.stack(x_train).T  # (N, K)
print("OOF shape:", x_train.shape)

In [ ]:
def auc_gpu_binned(y_true_cp, y_pred_cp, n_bins=4096):
    """
    Approx AUC for 1D preds on GPU via binning. Handles ties reasonably.
    y_true_cp: (N,) in {0,1}
    y_pred_cp: (N,) float
    """
    y = y_true_cp.astype(cp.int32)
    p = y_pred_cp.astype(cp.float32)

    pmin = cp.min(p)
    pmax = cp.max(p)
    denom = cp.maximum(pmax - pmin, cp.float32(1e-12))
    bins = cp.floor((p - pmin) / denom * (n_bins - 1)).astype(cp.int32)

    pos = cp.bincount(bins, weights=y, minlength=n_bins).astype(cp.float64)
    cnt = cp.bincount(bins, minlength=n_bins).astype(cp.float64)
    neg = cnt - pos

    cum_neg = cp.cumsum(neg)
    neg_before = cp.concatenate([cp.array([0.0]), cum_neg[:-1]])
    concordant = cp.sum(pos * neg_before)
    ties = cp.sum(pos * neg) * 0.5

    P = cp.sum(pos)
    N = cp.sum(neg)
    auc = (concordant + ties) / (P * N + 1e-12)
    return auc

def auc_gpu_binned_multi(y_true_cp, preds_cp_2d, n_bins=4096):
    """
    Approx AUC for (N, M) predictions at once.
    Returns (M,) AUCs.
    """
    M = preds_cp_2d.shape[1]
    out = cp.empty(M, dtype=cp.float64)
    for j in range(M):
        out[j] = auc_gpu_binned(y_true_cp, preds_cp_2d[:, j], n_bins=n_bins)
    return out

best_score = -1
best_index = -1

y = train[TARGET].astype(int)

true=y

for k, name in enumerate(files):
    s = roc_auc_score(true, x_train[:, k])
    print(f"AUC {s:.6f}  {name}")
    if s > best_score:
        best_score = s
        best_index = k

print("\nBest single model:", files[best_index], "AUC =", best_score)

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import cupy as cp
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# -----------------------------
# CONFIG & PATHS (From Version 2)
# -----------------------------

PATH = "/content/drive/MyDrive/Colab Notebooks/KAGGLE PS S6E3/OOF"
TARGET = "Churn"

# Ensemble Settings
USE_NEGATIVE_WGT = True
MAX_STEPS = 500  # Total iterations (Version 1 style)
TOL = 1e-7       # Improvement threshold
STEP_SIZE = 0.01

# -----------------------------
# 1) LOAD DATA & MODELS
# -----------------------------
# train = pd.read_csv(TRAIN_CSV)
# y = train[TARGET].astype(np.int32).values

files_names = ['cat', 'nn', 'xgb', 'lr', 'xgb_pseudo', 'xgb_te', 'xgb_v2',
               'torch_frame', 'realmlp', 'tabm', 'cat_v2', 'resnet',
               'lgbm_te', 'cat_te', 'tab_transformer', 'lgbm_te_goss',
               'gnn', 'xgb_res', 'hgb', 'ftt', 'bartz', 'tabm_pseudo', 'tabm_v2', 'cat_emb', 'hgb_te', 'cat_te_ordered', 'tabm_te', 'xlearn',
               'xgb_te_res', 'realmlp_32', 'realmlp_scratch', 'ftt_scratch', 'gatenet_scratch', 'danet_scratch', 'tabm_scratch']


x_train_list = []
print("Loading OOF files...")
for c in files_names:
    oof = np.load(f"{PATH}/{c}_oof_preds.npy")
    oof_ranked = (rankdata(oof) - 1.0) / (len(oof) - 1.0)
    x_train_list.append(oof_ranked)

x_train = np.column_stack(x_train_list)
N, K = x_train.shape
print(f"OOF Matrix: {x_train.shape}")

# -----------------------------
# 2) HELPERS (Version 1 AUC)
# -----------------------------
def multiple_roc_auc_scores(y_gpu, preds_gpu):
    """Exact GPU AUC using sorting (Version 1)"""
    n_pos = cp.sum(y_gpu)
    n_neg = y_gpu.size - n_pos
    # Argsort twice gives the rank
    ranks = cp.argsort(cp.argsort(preds_gpu, axis=0), axis=0) + 1
    aucs = (cp.sum(ranks[y_gpu == 1, :], axis=0) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return aucs

# -----------------------------
# 3) INITIALIZE
# -----------------------------
single_aucs = [roc_auc_score(y, x_train[:, i]) for i in range(K)]
best_idx = int(np.argmax(single_aucs))
best_score = float(single_aucs[best_idx])

print(f"\nBest single model: {files_names[best_idx]} | AUC: {best_score:.6f}")

x_gpu = cp.asarray(x_train, dtype=cp.float32)
y_gpu = cp.asarray(y, dtype=cp.int32)

# Weight grid (Version 1)
start_w = -0.50 if USE_NEGATIVE_WGT else 0.01
ww = cp.arange(start_w, 0.51, STEP_SIZE, dtype=cp.float32)

best_ensemble = x_gpu[:, best_idx].copy()
models_used = [best_idx]
weights_steps = [] # To store the 'w' used at each step

# -----------------------------
# 4) HILL CLIMB LOOP (Version 1 Logic)
# -----------------------------
print("\nStarting Hill Climb (Version 1 Logic)...")

for it in range(MAX_STEPS):
    # base = current_ens * (1-w)
    base = best_ensemble[:, None] * (1.0 - ww)[None, :]

    best_it_score = -1.0
    best_it_idx = -1
    best_it_w = 0.0
    best_it_ens = None

    for k in range(K):
        # Version 1 allows reusing models, so we check all K models
        cand = base + x_gpu[:, k][:, None] * ww[None, :]
        aucs = multiple_roc_auc_scores(y_gpu, cand)

        idx_w = int(cp.argmax(aucs).item())
        score_w = float(aucs[idx_w].item())

        if score_w > best_it_score:
            best_it_score = score_w
            best_it_idx = k
            best_it_w = float(ww[idx_w].item())
            best_it_ens = cand[:, idx_w]

    improve = best_it_score - best_score
    if improve < TOL:
        print(f"Stopped: Improvement {improve:.8f} < TOL")
        break

    best_score = best_it_score
    best_ensemble = best_it_ens
    models_used.append(best_it_idx)
    weights_steps.append(best_it_w)

    print(f"Step {it+1:3d}: AUC {best_score:.6f} | Add {files_names[best_it_idx]:15s} | w={best_it_w:.2f}")

    # Cleanup
    del base
    cp.get_default_memory_pool().free_all_blocks()

# -----------------------------
# 5) FINAL WEIGHT CALCULATION
# -----------------------------
# Version 1 uses recursive weighting: new_ens = old_ens*(1-w) + next_model*w
final_weights = np.zeros(K)
curr_w = 1.0

# Temporary array to track relative contribution
w_array = np.array([1.0])
for w in weights_steps:
    w_array = w_array * (1.0 - w)
    w_array = np.append(w_array, w)

# Map back to model indices
for idx, weight in zip(models_used, w_array):
    final_weights[idx] += weight

print("\n--- Final Ensemble Weights ---")
for i in np.argsort(-final_weights):
    if final_weights[i] > 1e-5 or final_weights[i] < -1e-5:
        print(f"{files_names[i]:15s} : {final_weights[i]:.6f}")

# Sanity Check
final_oof = np.zeros(N)
for i in range(K):
    final_oof += x_train[:, i] * final_weights[i]

print(f"\nFinal Exact AUC: {roc_auc_score(y, final_oof):.6f}")

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import cupy as cp
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# --- FIX KAGGLE AUTH ---
if os.path.exists('kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API token configured.")
else:
    print("❌ ERROR: kaggle.json not found! Please upload it to /content.")

# --- SHARED CONFIG ---
OOF_PATH = "/content/drive/MyDrive/Colab Notebooks/KAGGLE PS S6E3/OOF"
SUB_PATH = "/content/drive/MyDrive/Colab Notebooks/KAGGLE PS S6E3/SUB"
TARGET = "Churn"
USE_NEGATIVE_WGT = True
MAX_STEPS = 500
TOL = 1e-7
STEP_SIZE = 0.01

# original_files_names = [
#     'cat', 'nn', 'xgb', 'lr', 'xgb_pseudo', 'xgb_te', 'xgb_v2',
#     'torch_frame', 'realmlp', 'tabm', 'cat_v2', 'resnet',
#     'lgbm_te', 'cat_te', 'tab_transformer', 'lgbm_te_goss',
#     'gnn', 'xgb_res', 'hgb', 'ftt', 'bartz', 'tabm_pseudo',
#     'tabm_v2', 'cat_emb', 'hgb_te', 'cat_te_ordered', 'tabm_te',
#     'xlearn', 'xgb_te_res', 'realmlp_32'
# ]

original_files_names = [
    'cat', 'xgb', 'lr', 'xgb_pseudo', 'xgb_te', 'xgb_v2',
    'torch_frame', 'realmlp', 'tabm', 'cat_v2', 'resnet',
    'lgbm_te', 'cat_te', 'tab_transformer', 'lgbm_te_goss',
    'gnn', 'xgb_res', 'hgb', 'ftt', 'bartz', 'tabm_pseudo',
    'tabm_v2', 'cat_emb', 'hgb_te', 'cat_te_ordered', 'tabm_te',
    'xlearn', 'xgb_te_res', 'realmlp_32'
]


dont_drop_list = [    'cat', 'xgb', 'lr', 'xgb_pseudo', 'xgb_te', 'xgb_v2',
    'torch_frame', 'realmlp', 'tabm', 'cat_v2', 'resnet',
    'lgbm_te', 'cat_te', 'tab_transformer', 'lgbm_te_goss', 'xgb_res', 'hgb']
drop_list = [m for m in original_files_names if m not in dont_drop_list]


def multiple_roc_auc_scores(y_gpu, preds_gpu):
    n_pos = cp.sum(y_gpu)
    n_neg = y_gpu.size - n_pos
    ranks = cp.argsort(cp.argsort(preds_gpu, axis=0), axis=0) + 1
    aucs = (cp.sum(ranks[y_gpu == 1, :], axis=0) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return aucs

def rank_norm(series):
    r = rankdata(series)
    return (r - 1.0) / (len(r) - 1.0)

# --- AUTOMATION LOOP ---
for model_to_drop in drop_list:
    print(f"\n{'='*60}")
    print(f"RUNNING ENSEMBLE: DROPPING {model_to_drop}")
    print(f"{'='*60}")

    current_files = [f for f in original_files_names if f != model_to_drop]

    # 1) Load OOFs
    x_train_list = []
    for c in current_files:
        try:
            oof = np.load(f"{OOF_PATH}/{c}_oof_preds.npy")
            x_train_list.append(rank_norm(oof))
        except FileNotFoundError:
            print(f"Skipping {c}: OOF file not found.")
            continue

    if not x_train_list: continue

    x_train = np.column_stack(x_train_list)
    N, K = x_train.shape

    # 2) Initialize Hill Climb
    # y must be predefined in your notebook environment
    single_aucs = [roc_auc_score(y, x_train[:, i]) for i in range(K)]
    best_idx = int(np.argmax(single_aucs))
    best_score = float(single_aucs[best_idx])

    x_gpu = cp.asarray(x_train, dtype=cp.float32)
    y_gpu = cp.asarray(y, dtype=cp.int32)

    ww = cp.arange(-0.50 if USE_NEGATIVE_WGT else 0.01, 0.51, STEP_SIZE, dtype=cp.float32)
    best_ensemble = x_gpu[:, best_idx].copy()
    models_used = [best_idx]
    weights_steps = []

    # 3) Hill Climbing Loop
    for it in range(MAX_STEPS):
        base = best_ensemble[:, None] * (1.0 - ww)[None, :]
        best_it_score, best_it_idx, best_it_w, best_it_ens = -1.0, -1, 0.0, None

        for k in range(K):
            cand = base + x_gpu[:, k][:, None] * ww[None, :]
            aucs = multiple_roc_auc_scores(y_gpu, cand)
            idx_w = int(cp.argmax(aucs).item())
            score_w = float(aucs[idx_w].item())
            if score_w > best_it_score:
                best_it_score, best_it_idx, best_it_w, best_it_ens = score_w, k, float(ww[idx_w].item()), cand[:, idx_w]

        if (best_it_score - best_score) < TOL: break
        best_score, best_ensemble = best_it_score, best_it_ens
        models_used.append(best_it_idx)
        weights_steps.append(best_it_w)

    # 4) Calculate Final Weights
    final_weights_map = np.zeros(K)
    w_array = np.array([1.0])
    for w in weights_steps:
        w_array = w_array * (1.0 - w)
        w_array = np.append(w_array, w)
    for idx, weight in zip(models_used, w_array):
        final_weights_map[idx] += weight

    # 5) Generate Submission
    final_sub_df = pd.DataFrame()
    ensemble_preds = None

    for i, w in enumerate(final_weights_map):
        if abs(w) < 1e-6: continue
        name = current_files[i]

        if name == 'lama': fname = 'submission_lama.csv'
        elif name in ['cat', 'nn', 'lr', 'xgb', 'hgb', 'gnn', 'ftt', 'bartz', 'cat_emb', 'xlearn',
                      'tab_transformer', 'xgb_res', 'tabm_pseudo', 'tabm_v2', 'hgb_te',
                      'cat_te_ordered', 'tabm_te', 'xgb_te_res', 'realmlp_32']:
            fname = f"{name}_submission.csv"
        else: fname = f"{name}_sub.csv"

        full_path = os.path.join(SUB_PATH, fname)
        if os.path.exists(full_path):
            df = pd.read_csv(full_path)
            if final_sub_df.empty:
                final_sub_df['id'] = df['id']
                ensemble_preds = np.zeros(len(df))
            ensemble_preds += w * rank_norm(df[TARGET])

    # 6) Save and Submit
    if ensemble_preds is not None:
        final_sub_df[TARGET] = ensemble_preds
        sub_filename = f"sub_minus_{model_to_drop}.csv"
        final_sub_df.to_csv(sub_filename, index=False)

        print(f"Submitting: {sub_filename} | OOF AUC: {best_score:.6f}")
        message = f"Drop NN, {model_to_drop} | OOF AUC: {best_score:.6f}"

        # Check for daily limit errors
        sub_result = !kaggle competitions submit -c playground-series-s6e3 -f {sub_filename} -m "{message}"
        print("\n".join(sub_result))

        if any("Daily submission limit reached" in line for line in sub_result):
            print("🛑 Quota reached for today. Stopping loop.")
            break

    # Cleanup memory
    del x_gpu, y_gpu, best_ensemble, x_train, x_train_list
    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()